In [1]:
!pip install ultralytics easyocr ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 70.9 MB/s eta 0:00:00


In [15]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from ultralytics import YOLO
import cv2, numpy as np, random, base64, time, os, tempfile, subprocess
from IPython.display import display, HTML
from google.colab import files
import ipywidgets as widgets
from PIL import Image, ImageDraw, ImageFont

# ── Load model ────────────────────────────────────────────────
MODEL_PATH = Path("/content/drive/MyDrive/TrafficSign_YOLOv8/best.pt")
assert MODEL_PATH.exists(), f"Khong tim thay model: {MODEL_PATH}"
model = YOLO(str(MODEL_PATH))
class_names = model.names
print(f"Model: {MODEL_PATH.name} | {len(class_names)} lop")

# ── Bảng ánh xạ Tên Tiếng Việt Chi Tiết ────────────────────────
VIETNAMESE_NAMES = {
    'W.224': 'Đường người đi bộ cắt ngang',
    'W.205c': 'Đường giao nhau (Ngã tư)',
    'P.102': 'Cấm đi ngược chiều',
    'R.302a': 'Hướng phải đi theo: Rẽ phải',
    'W.205a': 'Đường giao nhau phía trước',
    'W.207': 'Giao nhau với đường không ưu tiên',
    'W.201a': 'Chỗ ngoặt nguy hiểm vòng bên trái',
    'P.123a': 'Cấm rẽ trái',
    'I.434a': 'Bến xe buýt',
    'R.303': 'Nơi giao nhau chạy theo vòng xuyến',
    'P.130': 'Cấm dừng và đỗ xe',
    'I.409': 'Chỗ quay xe',
    'R.415a': 'Biển gộp làn đường theo phương tiện',
    'W.245a': 'Đi chậm',
    'P.106a*Xe tải': 'Cấm ô tô tải',
    'W.203c': 'Đường bị hẹp cả hai bên',
    'P.117*': 'Hạn chế chiều cao',
    'P.124a*': 'Cấm quay đầu xe',
    'P.107': 'Cấm ô tô khách và ô tô tải',
    'P.124d': 'Cấm rẽ trái và quay đầu xe',
    'P.103a': 'Cấm ô tô',
    'W.203b': 'Đường bị hẹp bên phải',
    'W.221b': 'Đường có gờ giảm tốc',
    'P.111': 'Cấm xe gắn máy',
    'P.129': 'Kiểm tra',
    'S.505a*Xe máy': 'Biển phụ: Loại xe máy',
    'W.246a': 'Chú ý chướng ngại vật phía trước',
    'W.225': 'Trẻ em',
    'S.505a*Xe tải và công': 'Biển phụ: Xe tải và Container',
    'P.104': 'Cấm xe máy',
    'S.505a*Xe tải': 'Biển phụ: Xe tải',
    'Camera': 'Thiết bị giám sát giao thông',
    'P.123b': 'Cấm rẽ phải',
    'W.202b': 'Nhiều chỗ ngoặt nguy hiểm liên tiếp',
    'B.8a': 'Dừng lại',
    'P.137': 'Cấm đi thẳng và rẽ trái',
    'P.139': 'Cấm đi thẳng và rẽ phải',
    'W.205b': 'Đường giao nhau (Nhánh bên trái)',
    'P.127': 'Tốc độ tối đa',
    'R.301e': 'Hướng đi phải theo: Rẽ trái',
    'W.239b*': 'Đường cáp điện phía trên',
    'W.233': 'Nguy hiểm khác',
    'I.407a': 'Đường một chiều',
    'P.131a': 'Cấm đỗ xe',
    'P.124b1': 'Cấm ô tô quay đầu',
    'W.210': 'Giao nhau với đường sắt có rào chắn',
    'P.124c': 'Cấm ô tô rẽ trái và quay đầu',
    'W.201b': 'Chỗ ngoặt nguy hiểm vòng bên phải',
    'W.246c': 'Chú ý chướng ngại vật bên trái'
}

import easyocr
reader = easyocr.Reader(['en'], gpu=True, verbose=False)
print("EasyOCR san sang")

# ── Cài & load font hỗ trợ tiếng Việt có dấu ───────────────────
# load_default() của Pillow là font bitmap, KHÔNG hỗ trợ Unicode/dấu
# tiếng Việt -> phải đảm bảo có font TTF thật (DejaVu/Liberation/Noto)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "fonts-dejavu-core"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

FONT_PATHS = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
]
VNFONT = None
FONT_USED = None
for fp in FONT_PATHS:
    if os.path.exists(fp):
        VNFONT = ImageFont.truetype(fp, 9)
        FONT_USED = fp
        break

# Nếu apt-get không khả dụng (môi trường hạn chế quyền), tải font Noto
# Sans trực tiếp từ Google Fonts làm phương án dự phòng.
if VNFONT is None:
    try:
        import urllib.request
        backup_font_path = "/content/NotoSans-Regular.ttf"
        if not os.path.exists(backup_font_path):
            url = "https://github.com/google/fonts/raw/main/ofl/notosans/static/NotoSans-Regular.ttf"
            urllib.request.urlretrieve(url, backup_font_path)
        VNFONT = ImageFont.truetype(backup_font_path, 9)
        FONT_USED = backup_font_path
    except Exception as e:
        print(f"Khong tai duoc font du phong: {e}")

# Không cho fallback âm thầm sang bitmap font (đây là nguyên nhân gây
# mất dấu tiếng Việt trong bản gốc) -> báo lỗi rõ ràng nếu vẫn thất bại
assert VNFONT is not None, "Khong tim duoc font TTF ho tro Unicode. Kiem tra lai ket noi mang / quyen apt-get."
print(f"Da load font tieng Viet: {FONT_USED}")

# ── Config ────────────────────────────────────────────────────
CONF_THRESH   = 0.35
IOU_THRESH    = 0.45
IMG_SIZE      = 640
OCR_CONF      = 0.5
SPEED_CLASSES = {'P.127'}
VALID_SPEEDS  = set(range(5, 135, 5))

PALETTE = {}
def get_color(cls_name):
    if cls_name not in PALETTE:
        random.seed(hash(cls_name) % 2**32)
        PALETTE[cls_name] = tuple(random.randint(80, 230) for _ in range(3))
    return PALETTE[cls_name]

def preprocess_crop(crop):
    h, w = crop.shape[:2]
    if min(h, w) < 64:
        scale = 64 / min(h, w)
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray  = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    return cv2.cvtColor(clahe.apply(gray), cv2.COLOR_GRAY2BGR)

def crop_with_padding(img, x1, y1, x2, y2, pad=0.07):
    H, W = img.shape[:2]
    pw, ph = int((x2-x1)*pad), int((y2-y1)*pad)
    return img[max(0,y1-ph):min(H,y2+ph), max(0,x1-pw):min(W,x2+pw)]

def ocr_speed(crop):
    res = reader.readtext(preprocess_crop(crop), allowlist='0123456789', detail=1, paragraph=False)
    cands = sorted([(c,t) for _,t,c in res if c >= OCR_CONF and t.strip()], reverse=True)
    if not cands: return ''
    try:
        v = int(cands[0][1].strip())
        return str(v) if v in VALID_SPEEDS else ''
    except ValueError: return ''

def predict_frame(img_bgr):
    out = img_bgr.copy()
    res = model.predict(source=img_bgr, conf=CONF_THRESH, iou=IOU_THRESH, imgsz=IMG_SIZE, verbose=False)[0]
    detected_list = []
    if not res.boxes: return out, []

    pil_img = Image.fromarray(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil_img)

    for box in res.boxes:
        cls_id   = int(box.cls[0])
        cls_code = class_names[cls_id]
        cls_display_name = VIETNAMESE_NAMES.get(cls_code, cls_code)
        conf = float(box.conf[0])
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        color = get_color(cls_code)

        ocr_text = ''
        if cls_code in SPEED_CLASSES:
            ocr_text = ocr_speed(crop_with_padding(img_bgr, x1,y1,x2,y2))
            if not ocr_text: continue

        label = f"{cls_display_name} ({conf:.2f})" + (f" - {ocr_text} km/h" if ocr_text else "")
        detected_list.append(label)

        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)

        # Căn chỉnh text label
        tw, th = draw.textbbox((0, 0), label, font=VNFONT)[2:]
        draw.rectangle([x1, y1 - th - 10, x1 + tw + 10, y1], fill=color)
        draw.text((x1 + 5, y1 - th - 5), label, font=VNFONT, fill=(255, 255, 255))

    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR), detected_list

# ── UI Realtime ───────────────────────────────────────────────
display(HTML("""
<h3>Predict Realtime qua Webcam</h3>
<div style="display:flex; gap:20px;">
  <div>
    <button onclick="startCam()" style="padding:8px 18px;background:#1a73e8;color:#fff;border:none;border-radius:6px;cursor:pointer;">Bat Camera</button>
    <button onclick="stopCam()" style="padding:8px 18px;background:#d93025;color:#fff;border:none;border-radius:6px;cursor:pointer;">Tat Camera</button>
    <br><br>
    <img id="realtime-out" style="border:2px solid #444; border-radius:8px; width:640px; background:#111;" />
  </div>
  <div style="width:350px; padding:10px; border:1px solid #ccc; border-radius:8px; background:#f9f9f9;">
    <strong>Danh sach bien bao:</strong>
    <ul id="detection-list" style="font-size:13px; color:#333; padding-left:20px;"></ul>
  </div>
</div>

<script>
let stream = null, video = null, running = false;
async function startCam() {
  if (running) return;
  video = document.createElement('video');
  stream = await navigator.mediaDevices.getUserMedia({video: {width:640, height:480}});
  video.srcObject = stream; await video.play();
  running = true; captureLoop();
}
function stopCam() {
  running = false; if (stream) stream.getTracks().forEach(t => t.stop());
}
function captureLoop() {
  if (!running) return;
  const canvas = document.createElement('canvas');
  canvas.width = 640; canvas.height = 480;
  canvas.getContext('2d').drawImage(video, 0, 0);
  const b64 = canvas.toDataURL('image/jpeg', 0.8).split(',')[1];
  google.colab.kernel.invokeFunction('predict_and_return', [b64], {})
    .then(result => {
      const [imgB64, items] = result.data['application/json'];
      document.getElementById('realtime-out').src = 'data:image/jpeg;base64,' + imgB64;
      const listUI = document.getElementById('detection-list');
      listUI.innerHTML = items.map(i => `<li>${i}</li>`).join('');
      captureLoop();
    })
    .catch(e => { if (running) setTimeout(captureLoop, 100); });
}
</script>
"""))

import google.colab.output
from IPython.display import JSON

def predict_and_return(b64_frame: str):
    arr = np.frombuffer(base64.b64decode(b64_frame), dtype=np.uint8)
    frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if frame is None: return JSON(["", []])
    out, detected = predict_frame(frame)
    _, buf = cv2.imencode('.jpg', out, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return JSON([base64.b64encode(buf.tobytes()).decode(), detected])

google.colab.output.register_callback('predict_and_return', predict_and_return)

# ── UI Upload ────────────────────────────────────────────────
display(HTML("<h3>Predict tu anh hoac video upload</h3>"))
upload_btn = widgets.FileUpload(accept='.jpg,.jpeg,.png,.mp4', multiple=False)
run_btn = widgets.Button(description='Predict', button_style='primary')
out_widget = widgets.Output()
display(widgets.HBox([upload_btn, run_btn]), out_widget)

def on_run(b):
    out_widget.clear_output(wait=True)
    with out_widget:
        if not upload_btn.value: return
        name = list(upload_btn.value.keys())[0]
        raw = list(upload_btn.value.values())[0]['content']
        arr = np.frombuffer(raw, dtype=np.uint8)
        frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if frame is not None:
            res, items = predict_frame(frame)
            display(HTML(f"<b>Phat hien:</b> {', '.join(items) if items else 'Khong co'}"))
            _, buf = cv2.imencode('.jpg', res)
            display(HTML(f'<img src="data:image/jpeg;base64,{base64.b64encode(buf).decode()}" style="max-width:720px;"/>'))

run_btn.on_click(on_run)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model: best.pt | 21 lop
EasyOCR san sang
Da load font tieng Viet: /usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf


Output()